# Held-out 5×2 comparison on Kaggle T4

Same Tesla T4 as Stages A and B. Do not switch GPUs.

80 held-out inventories.

**Main (800):** five relays (full, headwise-32, H-OBF-32, headwise-64, H-OBF-64) × two access conditions (sender-only with filler, shared inventory).

**Predeclared sensitivity (240):** sender-only **without** filler for full, headwise-32, and OBF-32 only. That is 1,040 evaluations before separate controls.

**Controls (not in the 1,040):** direct-text and matched-mismatch.

The lead result is **sender-only with filler** versus actual relay size. Shared interprets that curve. No-filler at budget 32 checks whether filler changes the story. Bootstrap over the 80 examples.

The run resumes from `/kaggle/working/results_stage_c/evaluations.jsonl` if that file already exists. Download that folder during the run.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_c.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_c.py").is_file() and (cand / "data" / "test.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_c.py and data/test.jsonl. Re-upload the repo zip.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_c.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("test examples", sum(1 for _ in (REPO / "data" / "test.jsonl").open(encoding="utf-8")))

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Download /kaggle/working/results_stage_c during the run. Resume is on by default.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_c", "--split", "test"])
print("stage_c exit", rc)

In [ ]:
src = WORK / "results_stage_c"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_c" / "run"
print("reading", src)
for name in ("summary.json", "bootstrap.json", "manifest.json"):
    path = src / name
    if path.is_file():
        print("====", name)
        print(path.read_text(encoding="utf-8")[:5000])